# Decoder-Only GQA Trainer
Training a Decoder-Only Transformer with Grouped Query Attention (GQA) on GSM8K using PyTorch Lightning.

In [ ]:
!git clone https://github.com/ambideXtrous9/Transformer-from-Scratch.git /content/Transformer-from-Scratch
%cd /content/Transformer-from-Scratch/PLTrainerScripts
!pip install -r ../requirements.txt

In [ ]:
import os, sys

PROJECT_ROOT = os.path.dirname(os.getcwd())
sys.path.insert(0, PROJECT_ROOT)

import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from core.Embedding import get_tokenizer
from models.DecoderOnlyGQAModel import DecoderOnlyGQAModel
from pytorch_lightning.callbacks import ModelCheckpoint
from datasets import load_dataset
import config
from dotenv import load_dotenv
from pytorch_lightning.loggers import WandbLogger
import wandb

pl.seed_everything(config.SEED)
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

MAX_LENGTH = config.MAX_LENGTH

## Dataset

In [ ]:
class GSM8KDataset(Dataset):
    """
    Dataset for decoder-only (GPT-style) training on GSM8K.
    - Input: [BOS] + text
    - Labels: text + [EOS], with -100 for padding
    """
    def __init__(self, tokenizer, hf_dataset, max_length=256):
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.texts = []
        for sample in hf_dataset:
            text = f"Question: {sample['question']}\nAnswer: {sample['answer']}"
            self.texts.append(text)

        if tokenizer.bos_token is None:
            tokenizer.add_special_tokens({"bos_token": "<s>"})
        if tokenizer.eos_token is None:
            tokenizer.add_special_tokens({"eos_token": "</s>"})
        if tokenizer.pad_token is None:
            tokenizer.add_special_tokens({"pad_token": "<pad>"})

        self.pad_id = tokenizer.pad_token_id
        self.bos_id = tokenizer.bos_token_id
        self.eos_id = tokenizer.eos_token_id

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length - 2,
            return_tensors="pt",
            add_special_tokens=False
        )
        ids = enc["input_ids"].squeeze(0)

        input_ids = torch.cat([torch.tensor([self.bos_id]), ids], dim=0)
        labels = torch.cat([ids, torch.tensor([self.eos_id])], dim=0)

        if len(input_ids) < self.max_length:
            pad_len = self.max_length - len(input_ids)
            input_ids = torch.cat([input_ids, torch.full((pad_len,), self.pad_id)])
        else:
            input_ids = input_ids[:self.max_length]

        if len(labels) < self.max_length:
            pad_len = self.max_length - len(labels)
            labels = torch.cat([labels, torch.full((pad_len,), -100)])
        else:
            labels = labels[:self.max_length]

        return {
            "input_ids": input_ids.long(),
            "labels": labels.long()
        }

## Setup

In [ ]:
print("Loading GSM8K dataset from HuggingFace...")
gsm8k = load_dataset(config.DATASET_NAME, config.DATASET_CONFIG)
train_data = gsm8k["train"]
test_data = gsm8k["test"]

print(f"\nTrain samples: {len(train_data)}")
print(f"Test samples:  {len(test_data)}")

In [ ]:
tokenizer = get_tokenizer(config.TOKENIZER_NAME, add_pad_token_if_missing=True)
vocab_size = len(tokenizer)
pad_id = tokenizer.pad_token_id

train_dataset = GSM8KDataset(tokenizer, train_data, max_length=MAX_LENGTH)
val_dataset = GSM8KDataset(tokenizer, test_data, max_length=MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=config.TRAIN_BATCH_SIZE, shuffle=True, num_workers=config.NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=config.VAL_BATCH_SIZE, num_workers=config.NUM_WORKERS)

## Lightning Model

In [ ]:
model = DecoderOnlyGQAModel(
    vocab_size=vocab_size,
    d_model=config.D_MODEL,
    max_positions=MAX_LENGTH,
    num_layers=config.NUM_LAYERS,
    num_heads=config.NUM_HEADS,
    num_kv_heads=config.NUM_KV_HEADS,
    d_ff=config.D_FF,
    tokenizer=tokenizer,
    dropout=config.DROPOUT,
    pad_token_id=pad_id,
    lr=config.LEARNING_RATE
)

checkpoint_callback = ModelCheckpoint(
    dirpath = config.CHECKPOINTS["gqa"],
    filename = 'DecoderOnlyGQABestModel',
    save_top_k = 1,
    verbose = True,
    monitor = 'val_loss_epoch',
    mode = 'min'
)

## Trainer

In [ ]:
wandb_logger = WandbLogger(project=config.WANDB_PROJECT, name="DecoderOnly-GQA", log_model=False)
wandb_logger.experiment.config.update({
    "architecture": "DecoderOnly-GQA",
    "d_model": config.D_MODEL,
    "num_layers": config.NUM_LAYERS,
    "num_heads": config.NUM_HEADS,
    "num_kv_heads": config.NUM_KV_HEADS,
    "d_ff": config.D_FF,
    "dropout": config.DROPOUT,
    "learning_rate": config.LEARNING_RATE,
    "max_length": MAX_LENGTH,
    "batch_size": config.TRAIN_BATCH_SIZE,
    "max_epochs": config.MAX_EPOCHS,
})

trainer = pl.Trainer(
    max_epochs=config.MAX_EPOCHS,
    check_val_every_n_epoch=1,
    devices=-1,
    accelerator="gpu",
    callbacks=[checkpoint_callback],
    logger=wandb_logger
)

## Run Training

In [ ]:
trainer.fit(model, train_loader, val_loader)
wandb.finish()